In [ ]:
# 03 - Condition-Family Visualization

This notebook reproduces the condition-family voltage--SoC visualizations used in the manuscript.

It uses the representative-cell subset generated by:

`02_representative_cell_selection.ipynb`

Expected input file:

`outputs/02_representative_cell_selection/rep_filtered_byfile_aware.csv`

The notebook constructs 26 comparison families:

- 6 temperature-sweep families
- 12 charge cut-off current-sweep families
- 8 discharge-rate/rate-history-sweep families

The resulting outputs support the qualitative condition-family analysis reported in Section 4 of the manuscript.

In [ ]:
# ============================================================
# Setup and paths
# ============================================================

import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    display = print

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    files = None
    IN_COLAB = False

INPUT_REP_CSV = "outputs/02_representative_cell_selection/rep_filtered_byfile_aware.csv"
OUTPUT_DIR = "outputs/03_condition_family_visualization"
os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(INPUT_REP_CSV):
    raise FileNotFoundError(
        "Representative subset file not found. Run "
        "`02_representative_cell_selection.ipynb` first, or place "
        "`rep_filtered_byfile_aware.csv` at:\n"
        f"{INPUT_REP_CSV}"
    )

print("Input representative subset:")
print(INPUT_REP_CSV)

print("\nOutput directory:")
print(OUTPUT_DIR)

In [ ]:
# ============================================================
# Load representative subset
# ============================================================

rep_df = pd.read_csv(INPUT_REP_CSV)

required_cols = {
    "sample_int",
    "Condition_ID",
    "Temperature",
    "Rate",
    "Charge_Rate",
    "Step_Index",
    "CurrentA",
    "VoltageV",
    "SoC",
}

missing = sorted(required_cols - set(rep_df.columns))
if missing:
    raise ValueError(f"Missing required columns in representative subset: {missing}")

print("Representative subset loaded.")
print(f"Rows: {len(rep_df):,}")
print(f"Unique representative sample IDs: {rep_df['sample_int'].nunique()}")
print(f"Covered conditions: {rep_df['Condition_ID'].nunique()} / 24")

display(rep_df.head())

In [ ]:
# ============================================================
# Condition lookup and factor levels
# ============================================================
# Column convention in the processed files:
#
#   Rate         -> charge cut-off current level: C/5 or C/40
#   Charge_Rate -> discharge C-rate: 0.7C, 1C, or 2C

T_LEVELS = [10, 25, 45, 60]
R_LEVELS = ["C/5", "C/40"]
DR_LEVELS = ["0.7C", "1C", "2C"]

def build_condition_lookup():
    rows = []
    cid = 1

    for T in T_LEVELS:
        for DR in DR_LEVELS:
            for R in R_LEVELS:
                rows.append((cid, T, R, DR))
                cid += 1

    return pd.DataFrame(
        rows,
        columns=["Condition_ID", "Temperature", "Rate", "Charge_Rate"]
    )

COND_LUT = build_condition_lookup()

display(COND_LUT)

In [ ]:
# ============================================================
# Use discharge records only
# ============================================================

rep_discharge = rep_df[
    (rep_df["Step_Index"] == 5) &
    (rep_df["CurrentA"] < 0)
].copy()

print("Discharge-only representative subset:")
print(f"Rows: {len(rep_discharge):,}")
print(f"Unique representative sample IDs: {rep_discharge['sample_int'].nunique()}")
print(f"Covered conditions: {rep_discharge['Condition_ID'].nunique()} / 24")

display(
    rep_discharge[
        ["sample_int", "Condition_ID", "Temperature", "Rate",
         "Charge_Rate", "Step_Index", "CurrentA", "VoltageV", "SoC"]
    ].head()
)

In [ ]:
# ============================================================
# SoC grid and sample-level curve construction
# ============================================================

SOC_MIN = 5.0
SOC_MAX = 95.0
SOC_STEP = 1.0

soc_grid = np.arange(SOC_MIN, SOC_MAX + 1e-9, SOC_STEP)

def sample_curve(group):
    """
    Construct a voltage--SoC curve for one retained sample-condition pair.

    The curve is interpolated on a conservative 5--95% SoC grid with 1%
    spacing. This visualization grid avoids unstable terminal endpoints and
    corresponds to the condition-family curves used in the manuscript.
    """
    g = (
        group[["SoC", "VoltageV"]]
        .dropna()
        .copy()
    )

    if g.empty:
        return pd.DataFrame({"SoC": soc_grid, "VoltageV_med": np.nan})

    g["SoC"] = pd.to_numeric(g["SoC"], errors="coerce")
    g["VoltageV"] = pd.to_numeric(g["VoltageV"], errors="coerce")
    g = g.dropna().sort_values("SoC")

    if g.empty:
        return pd.DataFrame({"SoC": soc_grid, "VoltageV_med": np.nan})

    # Collapse duplicate SoC values before interpolation.
    g_med = (
        g.groupby("SoC", as_index=False)["VoltageV"]
        .median()
        .sort_values("SoC")
    )

    if g_med["SoC"].nunique() < 2:
        return pd.DataFrame({"SoC": soc_grid, "VoltageV_med": np.nan})

    v_interp = np.interp(
        soc_grid,
        g_med["SoC"].values,
        g_med["VoltageV"].values,
        left=np.nan,
        right=np.nan
    )

    return pd.DataFrame({"SoC": soc_grid, "VoltageV_med": v_interp})

In [ ]:
# ============================================================
# Build sample-condition curves
# ============================================================

curves = []

for (sid, cid), grp in rep_discharge.groupby(["sample_int", "Condition_ID"]):
    sc = sample_curve(grp)
    sc["sample_int"] = sid
    sc["Condition_ID"] = cid
    curves.append(sc)

curves = pd.concat(curves, ignore_index=True)

print("Sample-condition curves built.")
print(f"Rows: {len(curves):,}")
print(f"Unique sample IDs: {curves['sample_int'].nunique()}")
print(f"Covered conditions: {curves['Condition_ID'].nunique()} / 24")
print(f"SoC grid: {SOC_MIN:.0f}--{SOC_MAX:.0f}% with {SOC_STEP:.0f}% step")

display(curves.head())

In [ ]:
# ============================================================
# Define 26 comparison families
# ============================================================

families = []

# 6 temperature-sweep families:
# fixed charge cut-off current and discharge-rate/rate-history level;
# varying temperature.
for R in R_LEVELS:
    for DR in DR_LEVELS:
        families.append({
            "type": "T_sweep",
            "fixed": {"Rate": R, "Charge_Rate": DR},
            "vary": {"Temperature": T_LEVELS},
        })

# 12 charge cut-off current-sweep families:
# fixed temperature and discharge-rate/rate-history level;
# varying charge cut-off current.
for T in T_LEVELS:
    for DR in DR_LEVELS:
        families.append({
            "type": "Rate_sweep",
            "fixed": {"Temperature": T, "Charge_Rate": DR},
            "vary": {"Rate": R_LEVELS},
        })

# 8 discharge-rate/rate-history-sweep families:
# fixed temperature and charge cut-off current;
# varying discharge-rate/rate-history level.
for T in T_LEVELS:
    for R in R_LEVELS:
        families.append({
            "type": "DR_sweep",
            "fixed": {"Temperature": T, "Rate": R},
            "vary": {"Charge_Rate": DR_LEVELS},
        })

assert len(families) == 26

def cond_ids_for(fixed_values):
    tmp = COND_LUT.copy()
    for key, value in fixed_values.items():
        tmp = tmp[tmp[key] == value]
    return tmp["Condition_ID"].tolist()

print(f"Number of comparison families: {len(families)}")
display(pd.DataFrame(families).head(10))

In [ ]:
# ============================================================
# Generate family-level curves, plots, and CSV outputs
# ============================================================

family_index_rows = []
cond_counts = rep_discharge.groupby("Condition_ID")["sample_int"].nunique().to_dict()

for idx, fam in enumerate(families, start=1):
    ftype = fam["type"]
    fixed = fam["fixed"]
    vary = fam["vary"]

    v_key = list(vary.keys())[0]
    v_vals = vary[v_key]

    cond_rows = []

    for vv in v_vals:
        fixed_now = fixed.copy()
        fixed_now[v_key] = vv

        cid_list = cond_ids_for(fixed_now)

        for cid in cid_list:
            cond_rows.append((vv, cid))

    fam_dir = os.path.join(OUTPUT_DIR, f"family_{idx:02d}_{ftype}")
    os.makedirs(fam_dir, exist_ok=True)

    plt.figure(figsize=(8, 5))

    per_condition_tables = []

    for vv, cid in cond_rows:
        cc = curves[curves["Condition_ID"] == cid].copy()

        if cc.empty:
            continue

        grouped = cc.groupby("SoC")["VoltageV_med"]

        agg = pd.DataFrame({
            "SoC": sorted(cc["SoC"].dropna().unique())
        })

        agg["V_median"] = grouped.median().reindex(agg["SoC"]).values
        agg["V_q25"] = grouped.quantile(0.25).reindex(agg["SoC"]).values
        agg["V_q75"] = grouped.quantile(0.75).reindex(agg["SoC"]).values

        x = agg["SoC"].astype(float).values
        y = agg["V_median"].astype(float).values
        y25 = agg["V_q25"].astype(float).values
        y75 = agg["V_q75"].astype(float).values

        label = f"{v_key}={vv} (CID={cid}, n={cond_counts.get(cid, 0)})"

        plt.plot(x, y, label=label)

        try:
            plt.fill_between(x, y25, y75, alpha=0.15)
        except Exception:
            pass

        agg["Condition_ID"] = cid
        agg[v_key] = vv
        agg["n_samples_in_condition"] = cond_counts.get(cid, 0)

        per_condition_tables.append(agg)

    plt.title(f"{ftype} | fixed: {fixed}")
    plt.xlabel("SoC (%)")
    plt.ylabel("Voltage (V)")
    plt.grid(True)

    if per_condition_tables:
        plt.legend(fontsize=8)

    plt.tight_layout()

    fig_path = os.path.join(fam_dir, "plot_v_soc.png")
    plt.savefig(fig_path, dpi=300)
    plt.close()

    if per_condition_tables:
        fam_tab = pd.concat(per_condition_tables, ignore_index=True)
    else:
        fam_tab = pd.DataFrame(
            columns=[
                "SoC", "V_median", "V_q25", "V_q75",
                "Condition_ID", v_key, "n_samples_in_condition"
            ]
        )

    fam_csv = os.path.join(fam_dir, "family_curves.csv")
    fam_tab.to_csv(fam_csv, index=False)

    cond_summary_rows = []

    for vv, cid in cond_rows:
        row = {
            "Condition_ID": cid,
            v_key: vv,
            "n_samples_in_condition": cond_counts.get(cid, 0),
        }
        row.update(fixed)
        cond_summary_rows.append(row)

    cond_summary = pd.DataFrame(cond_summary_rows)
    cond_csv = os.path.join(fam_dir, "family_summary.csv")
    cond_summary.to_csv(cond_csv, index=False)

    family_index_rows.append({
        "family_index": idx,
        "type": ftype,
        "fixed": str(fixed),
        "vary_key": v_key,
        "vary_values": str(v_vals),
        "plot_path": os.path.relpath(fig_path, OUTPUT_DIR),
        "curves_csv": os.path.relpath(fam_csv, OUTPUT_DIR),
        "summary_csv": os.path.relpath(cond_csv, OUTPUT_DIR),
    })

family_index = pd.DataFrame(family_index_rows)

family_index_csv = os.path.join(OUTPUT_DIR, "family_index.csv")
family_index.to_csv(family_index_csv, index=False)

curves_csv = os.path.join(OUTPUT_DIR, "sample_condition_curves_5_95_soc_grid.csv")
curves.to_csv(curves_csv, index=False)

print("Family visualization outputs generated.")
print(f"Output directory: {OUTPUT_DIR}")

print("\nFamily index:")
display(family_index.head(10))

In [ ]:
# ============================================================
# Create ZIP archive
# ============================================================

zip_path = "outputs_03_condition_family_visualization.zip"

if os.path.exists(zip_path):
    os.remove(zip_path)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, dirs, files_in_dir in os.walk(OUTPUT_DIR):
        for file_name in files_in_dir:
            full_path = os.path.join(root, file_name)
            arcname = os.path.relpath(full_path, ".")
            z.write(full_path, arcname=arcname)

print(f"ZIP archive created: {zip_path}")

if IN_COLAB and files is not None:
    try:
        files.download(zip_path)
    except Exception:
        pass

In [ ]:
# ============================================================
# Final summary
# ============================================================

print("=== Condition-Family Visualization Summary ===")
print(f"Number of families: {len(families)}")
print(f"Representative discharge records: {len(rep_discharge):,}")
print(f"Sample-condition curve rows: {len(curves):,}")
print(f"Covered conditions in curves: {curves['Condition_ID'].nunique()} / 24")
print(f"Output directory: {OUTPUT_DIR}")
print(f"ZIP archive: {zip_path}")

print("\nFirst rows of family index:")
display(family_index.head(10))

In [ ]:
## Expected outputs

This notebook generates the condition-family visualization outputs used for the manuscript's qualitative condition-family analysis.

Main outputs:

- `outputs/03_condition_family_visualization/family_index.csv`
- `outputs/03_condition_family_visualization/sample_condition_curves_5_95_soc_grid.csv`
- `outputs/03_condition_family_visualization/family_*/plot_v_soc.png`
- `outputs/03_condition_family_visualization/family_*/family_curves.csv`
- `outputs/03_condition_family_visualization/family_*/family_summary.csv`
- `outputs_03_condition_family_visualization.zip`

The 5--95% SoC visualization grid is used only for condition-family plotting. It is distinct from the 0--90% inferential domain and from the 0--100% deployment LUT grid discussed in the manuscript.